# 11 run infercnvpy

**Purpose:** Run infercnvpy across the predefined robustness strategies.

**Original analysis notebook:** `11_run_infercnvpy_CNV_HVG8000_r0_6_strategies_v2_fixed_cnv_columns.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.


# 27 — Run infercnvpy CNV inference on HVG8000 r0.6 CNV strategies

This notebook runs **infercnvpy** on the CNV-ready sampled strategy objects prepared in the previous step.

Main principles:

- Use the **full-gene CNV inputs** prepared from the 15,176-gene object.
- Use sampled reference/query strategy objects to keep memory under control.
- Run strategies sequentially to avoid RAM crashes.
- Treat CNV calls as **evidence**, not final malignant labels.
- Use `core_conservative` as the main reference strategy, and the others as robustness/review runs.
- Do not calculate per-gene CNV layers to reduce memory usage.

Expected input manifest:

```text
${GLIOMA_PROJECT_ROOT}/metadata/CNV_strategy_input_manifest_HVG8000_r0_6.tsv
```


**v2 fix:** This version recognizes `cnv_chr`, `cnv_start`, and `cnv_end` as valid gene-position columns and converts them to the `chromosome/start/end` fields required downstream.

In [ ]:

# =========================
# Imports and global settings
# =========================

from pathlib import Path
import os
import gc
import time
import json
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import matplotlib.pyplot as plt

try:
    import infercnvpy as cnv
except Exception as e:
    raise ImportError(
        "infercnvpy could not be imported. Install it in the active environment first. "
        "Example: pip install infercnvpy"
    ) from e

warnings.filterwarnings("ignore")

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white")

# Keep BLAS/threading conservative for a 16 GB RAM machine.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")

print("scanpy:", sc.__version__)
print("infercnvpy:", getattr(cnv, "__version__", "unknown"))


In [ ]:

# =========================
# Paths and parameters
# =========================

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
RAW_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
RESULTS_DIR = PROJECT_DIR / "results"

CNV_INPUT_DIR = RAW_DIR / "integrated" / "CNV_inputs"
CNV_RESULT_DIR = RAW_DIR / "integrated" / "CNV_results" / "infercnvpy_HVG8000_r0_6"
CNV_FIGURE_DIR = FIGURES_DIR / "CNV_infercnvpy_HVG8000_r0_6"

for d in [METADATA_DIR, FIGURES_DIR, RESULTS_DIR, CNV_RESULT_DIR, CNV_FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STRATEGY_MANIFEST_TSV = METADATA_DIR / "CNV_strategy_input_manifest_HVG8000_r0_6.tsv"
GENE_ORDER_TSV = METADATA_DIR / "CNV_gene_order_gencode_v49_matched_header_HVG8000_r0_6.tsv"

# Run order: main strategy first, robustness/review strategies after.
RUN_STRATEGIES = [
    "core_conservative",
    "core_plus_lymphoid",
    "extended_myeloid",
    "review_cycling",
    "review_ambiguous",
]

# infercnvpy parameters.
# Conservative memory settings for 16 GB RAM.
WINDOW_SIZE = 100
STEP = 10
DYNAMIC_THRESHOLD = 1.5
LFC_CLIP = 3
CHUNKSIZE = 1000
N_JOBS = 2
EXCLUDE_CHROMOSOMES = ("chrX", "chrY", "X", "Y", "chrM", "chrMT", "MT", "M")

# CNV graph / clustering parameters.
CNV_PCA_N_COMPS = 30
CNV_NEIGHBORS = 15
CNV_LEIDEN_RESOLUTION = 0.5
RANDOM_STATE = 42

# Main biological clustering key transferred from scVI.
CLUSTER_KEY = "leiden_scvi_main"
CNV_GROUP_KEY = "cnv_group"
REFERENCE_CAT = "reference"
QUERY_CAT = "query"

OVERWRITE = True

print("Strategy manifest:", STRATEGY_MANIFEST_TSV)
print("CNV result dir:", CNV_RESULT_DIR)
print("CNV figure dir:", CNV_FIGURE_DIR)


In [ ]:

# =========================
# Utility functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")


def matrix_values_sample(X, n_rows=1000, n_cols=1000):
    xr = X[:min(X.shape[0], n_rows), :min(X.shape[1], n_cols)]
    if sp.issparse(xr):
        vals = xr.data
    else:
        vals = np.asarray(xr).ravel()
    vals = vals[np.isfinite(vals)]
    return vals


def matrix_is_integer_like(X):
    vals = matrix_values_sample(X)
    if vals.size == 0:
        return True
    return bool(np.allclose(vals, np.round(vals)))


def has_negative_values(X):
    vals = matrix_values_sample(X)
    if vals.size == 0:
        return False
    return bool(np.any(vals < 0))


def natural_chrom_sort_key(chrom):
    c = str(chrom).replace("chr", "")
    if c.isdigit():
        return (0, int(c))
    if c == "X":
        return (1, 23)
    if c == "Y":
        return (1, 24)
    if c in ["M", "MT"]:
        return (2, 25)
    return (3, c)


def standardize_gene_position_columns(adata):
    # Ensure adata.var has chromosome/start/end columns required by infercnvpy.
    var = adata.var

    # The CNV input-preparation notebook stores genomic positions as
    # cnv_chr / cnv_start / cnv_end. We also support common aliases
    # so this runner remains compatible with future objects.
    chrom_candidates = [
        "chromosome", "chrom", "chr", "seqnames",
        "gene_chr", "gene_chromosome",
        "cnv_chr", "cnv_chrom", "cnv_chromosome"
    ]
    start_candidates = [
        "start", "gene_start", "tx_start", "begin",
        "cnv_start", "cnv_gene_start"
    ]
    end_candidates = [
        "end", "gene_end", "tx_end", "stop",
        "cnv_end", "cnv_gene_end"
    ]

    def find_col(candidates):
        for c in candidates:
            if c in var.columns:
                return c
        return None

    chrom_col = find_col(chrom_candidates)
    start_col = find_col(start_candidates)
    end_col = find_col(end_candidates)

    if chrom_col is None or start_col is None or end_col is None:
        raise ValueError(
            "Gene position columns not found in adata.var. "
            f"Available var columns: {list(var.columns)}"
        )

    if chrom_col != "chromosome":
        adata.var["chromosome"] = adata.var[chrom_col].astype(str)
    else:
        adata.var["chromosome"] = adata.var["chromosome"].astype(str)

    # Normalize chromosome names to chr-prefixed style where possible.
    def normalize_chr(x):
        x = str(x)
        if x.startswith("chr"):
            return x
        if x in ["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","M","MT"]:
            return "chr" + x
        return x

    adata.var["chromosome"] = adata.var["chromosome"].map(normalize_chr)
    adata.var["start"] = pd.to_numeric(adata.var[start_col], errors="coerce")
    adata.var["end"] = pd.to_numeric(adata.var[end_col], errors="coerce")

    valid = (
        adata.var["chromosome"].notna()
        & adata.var["start"].notna()
        & adata.var["end"].notna()
    )

    if valid.sum() < adata.n_vars:
        print(f"[FILTER] Removing genes with missing genomic position: {adata.n_vars - int(valid.sum())}")
        adata = adata[:, valid.values].copy()

    sort_df = pd.DataFrame({
        "gene": adata.var_names.astype(str),
        "chromosome": adata.var["chromosome"].astype(str).values,
        "start": adata.var["start"].astype(float).values,
        "end": adata.var["end"].astype(float).values,
    }, index=adata.var_names)
    sort_df["chrom_sort"] = sort_df["chromosome"].map(natural_chrom_sort_key)
    sort_df = sort_df.sort_values(["chrom_sort", "start", "end", "gene"])
    adata = adata[:, sort_df.index].copy()

    return adata


def compute_abs_cnv_score(adata, cnv_key="X_cnv"):
    if cnv_key not in adata.obsm:
        raise KeyError(f"{cnv_key} not found in adata.obsm")
    X = adata.obsm[cnv_key]
    if sp.issparse(X):
        score = np.asarray(np.abs(X).mean(axis=1)).ravel()
    else:
        score = np.mean(np.abs(np.asarray(X)), axis=1)
    return score


def top_fraction(series):
    counts = series.astype(str).value_counts(dropna=False)
    if counts.empty:
        return pd.Series({"top": "NA", "top_n": 0, "top_fraction": np.nan, "n_unique": 0})
    return pd.Series({
        "top": counts.index[0],
        "top_n": int(counts.iloc[0]),
        "top_fraction": float(counts.iloc[0] / counts.sum()),
        "n_unique": int(counts.shape[0]),
    })


def robust_ref_thresholds(ref_scores):
    ref_scores = np.asarray(ref_scores, dtype=float)
    ref_scores = ref_scores[np.isfinite(ref_scores)]
    med = float(np.median(ref_scores))
    mad = float(np.median(np.abs(ref_scores - med)))
    if mad == 0:
        mad = float(np.std(ref_scores))
    if mad == 0:
        mad = 1e-9
    q95 = float(np.quantile(ref_scores, 0.95))
    q99 = float(np.quantile(ref_scores, 0.99))
    moderate = max(q95, med + 2 * 1.4826 * mad)
    high = max(q99, med + 3 * 1.4826 * mad)
    return {
        "reference_median": med,
        "reference_mad": mad,
        "reference_q95": q95,
        "reference_q99": q99,
        "moderate_threshold": float(moderate),
        "high_threshold": float(high),
    }


def call_cluster(mean_score, median_score, thresholds):
    score = max(float(mean_score), float(median_score))
    if score >= thresholds["high_threshold"]:
        return "CNV_high_candidate"
    if score >= thresholds["moderate_threshold"]:
        return "CNV_intermediate_candidate"
    return "CNV_low_or_reference_like"


In [ ]:

# =========================
# Read strategy manifest and validate input files
# =========================

if not STRATEGY_MANIFEST_TSV.exists():
    raise FileNotFoundError(STRATEGY_MANIFEST_TSV)

manifest = pd.read_csv(STRATEGY_MANIFEST_TSV, sep="\t")
manifest = manifest[manifest["strategy"].isin(RUN_STRATEGIES)].copy()
manifest["strategy"] = pd.Categorical(manifest["strategy"], categories=RUN_STRATEGIES, ordered=True)
manifest = manifest.sort_values("strategy").reset_index(drop=True)

rows = []
for _, r in manifest.iterrows():
    h5 = Path(r["sampled_h5ad"])
    rows.append({
        "strategy": r["strategy"],
        "sampled_h5ad": str(h5),
        "exists": h5.exists(),
        "size_mb": round(h5.stat().st_size / 1024**2, 3) if h5.exists() else np.nan,
        "reference_clusters": r["reference_clusters"],
        "query_clusters": r["query_clusters"],
        "n_sampled_reference_cells": r["n_sampled_reference_cells"],
        "n_sampled_query_cells": r["n_sampled_query_cells"],
        "n_sampled_total_cells": r["n_sampled_total_cells"],
    })

readiness = pd.DataFrame(rows)
READINESS_TSV = METADATA_DIR / "infercnvpy_run_readiness_HVG8000_r0_6.tsv"
write_tsv_replace(readiness, READINESS_TSV)

display(readiness)

if not readiness["exists"].all():
    missing = readiness.loc[~readiness["exists"], "sampled_h5ad"].tolist()
    raise FileNotFoundError(f"Missing strategy h5ad files: {missing}")


In [ ]:

# =========================
# Main per-strategy runner
# =========================

def run_one_infercnvpy_strategy(strategy_row):
    strategy = str(strategy_row["strategy"])
    input_h5ad = Path(strategy_row["sampled_h5ad"])

    print("\n" + "=" * 100)
    print(f"[START] {strategy}")
    print(f"Input: {input_h5ad}")
    t0 = time.time()

    out_prefix = f"infercnvpy_{strategy}_HVG8000_r0_6"
    result_h5ad = CNV_RESULT_DIR / f"{out_prefix}_result.h5ad"
    cell_scores_tsv = METADATA_DIR / f"{out_prefix}_cell_scores.tsv"
    cluster_summary_tsv = METADATA_DIR / f"{out_prefix}_cluster_summary.tsv"
    cnv_leiden_summary_tsv = METADATA_DIR / f"{out_prefix}_cnv_leiden_summary.tsv"
    run_summary_tsv = METADATA_DIR / f"{out_prefix}_run_summary.tsv"

    adata = sc.read_h5ad(input_h5ad)
    print(adata)

    if CNV_GROUP_KEY not in adata.obs.columns:
        raise KeyError(f"{CNV_GROUP_KEY} not found in adata.obs. Available: {list(adata.obs.columns)}")
    if CLUSTER_KEY not in adata.obs.columns:
        raise KeyError(f"{CLUSTER_KEY} not found in adata.obs. Available: {list(adata.obs.columns)}")

    adata.obs[CNV_GROUP_KEY] = adata.obs[CNV_GROUP_KEY].astype(str)
    adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype(str)

    # Ensure gene positions are present and sorted.
    adata = standardize_gene_position_columns(adata)

    # Record input count status before log-normalization.
    input_integer_like = matrix_is_integer_like(adata.X)
    input_has_negative = has_negative_values(adata.X)
    if input_has_negative:
        raise ValueError(f"Negative values found in X for {strategy}. CNV input should not contain negative counts/log values.")

    # infercnvpy expects normalized/log-transformed expression.
    # The sampled input X is raw counts from the previous notebook, so normalize/log1p if it looks count-like.
    preprocessing = "unknown"
    if input_integer_like:
        print("[PREPROCESS] X looks like raw counts; applying normalize_total + log1p.")
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        preprocessing = "normalize_total_1e4_log1p_from_raw_counts_X"
    else:
        print("[PREPROCESS] X does not look integer-like; assuming it is already normalized/log-transformed.")
        preprocessing = "used_existing_X_no_normalization"

    # Run infercnvpy. Do not calculate per-gene values to save memory.
    cnv.tl.infercnv(
        adata,
        reference_key=CNV_GROUP_KEY,
        reference_cat=REFERENCE_CAT,
        lfc_clip=LFC_CLIP,
        window_size=WINDOW_SIZE,
        step=STEP,
        dynamic_threshold=DYNAMIC_THRESHOLD,
        exclude_chromosomes=EXCLUDE_CHROMOSOMES,
        chunksize=CHUNKSIZE,
        n_jobs=N_JOBS,
        layer=None,
        key_added="cnv",
        calculate_gene_values=False,
    )

    if "X_cnv" not in adata.obsm:
        raise KeyError("infercnvpy did not create adata.obsm['X_cnv']")

    # Manual per-cell CNV burden. This is independent of cnv clustering.
    adata.obs["cnv_abs_mean_score"] = compute_abs_cnv_score(adata, "X_cnv")

    # CNV embedding / clustering. These are helpful diagnostics, not final labels.
    try:
        cnv.tl.pca(
            adata,
            use_rep="cnv",
            key_added="cnv_pca",
            n_comps=CNV_PCA_N_COMPS,
            svd_solver="arpack",
            zero_center=False,
        )
        cnv.pp.neighbors(
            adata,
            use_rep="cnv_pca",
            key_added="cnv_neighbors",
            n_neighbors=CNV_NEIGHBORS,
            random_state=RANDOM_STATE,
        )
        cnv.tl.leiden(
            adata,
            neighbors_key="cnv_neighbors",
            key_added="cnv_leiden",
            resolution=CNV_LEIDEN_RESOLUTION,
            random_state=RANDOM_STATE,
        )
        cnv.tl.umap(
            adata,
            neighbors_key="cnv_neighbors",
            key_added="cnv_umap",
            random_state=RANDOM_STATE,
        )
        cnv.tl.cnv_score(
            adata,
            groupby="cnv_leiden",
            use_rep="cnv",
            key_added="cnv_cluster_score",
        )
    except Exception as e:
        print("[WARNING] CNV PCA/neighbors/leiden/UMAP failed:", repr(e))

    ref_scores = adata.obs.loc[adata.obs[CNV_GROUP_KEY] == REFERENCE_CAT, "cnv_abs_mean_score"].astype(float).values
    thresholds = robust_ref_thresholds(ref_scores)

    # Cluster-level summary based on the biological scVI cluster key.
    summary_rows = []
    meta_cols = [
        "core_dataset", "condition", "patient_id", "sample_id",
        "annotation_marker_based_HVG8000_r0_6",
        "lineage_marker_based_HVG8000_r0_6",
        "annotation_confidence_HVG8000_r0_6",
        "cnv_role_HVG8000_r0_6",
        "bias_status_HVG8000_r0_6",
    ]
    present_meta_cols = [c for c in meta_cols if c in adata.obs.columns]

    for cl, df in adata.obs.groupby(CLUSTER_KEY, observed=True):
        scores = df["cnv_abs_mean_score"].astype(float).values
        groups = df[CNV_GROUP_KEY].astype(str).value_counts()
        main_group = groups.index[0] if len(groups) else "NA"
        row = {
            "strategy": strategy,
            "cluster": str(cl),
            "n_cells": int(df.shape[0]),
            "cnv_group_majority": main_group,
            "cnv_group_majority_fraction": float(groups.iloc[0] / groups.sum()) if len(groups) else np.nan,
            "mean_cnv_abs_score": float(np.mean(scores)),
            "median_cnv_abs_score": float(np.median(scores)),
            "q75_cnv_abs_score": float(np.quantile(scores, 0.75)),
            "q90_cnv_abs_score": float(np.quantile(scores, 0.90)),
            "reference_median": thresholds["reference_median"],
            "reference_q95": thresholds["reference_q95"],
            "reference_q99": thresholds["reference_q99"],
            "moderate_threshold": thresholds["moderate_threshold"],
            "high_threshold": thresholds["high_threshold"],
            "cnv_call": call_cluster(np.mean(scores), np.median(scores), thresholds),
        }
        # Add robust z-like effect based on reference median/MAD.
        row["robust_effect_vs_reference_median"] = float((row["median_cnv_abs_score"] - thresholds["reference_median"]) / (1.4826 * thresholds["reference_mad"] + 1e-9))

        for col in present_meta_cols:
            vc = df[col].astype(str).value_counts(dropna=False)
            row[f"top_{col}"] = vc.index[0] if len(vc) else "NA"
            row[f"top_{col}_fraction"] = float(vc.iloc[0] / vc.sum()) if len(vc) else np.nan

        summary_rows.append(row)

    cluster_summary = pd.DataFrame(summary_rows)
    write_tsv_replace(cluster_summary, cluster_summary_tsv)

    # CNV-leiden summary if available.
    if "cnv_leiden" in adata.obs.columns:
        cnv_leiden_rows = []
        for cl, df in adata.obs.groupby("cnv_leiden", observed=True):
            scores = df["cnv_abs_mean_score"].astype(float).values
            row = {
                "strategy": strategy,
                "cnv_leiden": str(cl),
                "n_cells": int(df.shape[0]),
                "mean_cnv_abs_score": float(np.mean(scores)),
                "median_cnv_abs_score": float(np.median(scores)),
                "q90_cnv_abs_score": float(np.quantile(scores, 0.90)),
            }
            for col in [CNV_GROUP_KEY, CLUSTER_KEY, "core_dataset", "condition", "patient_id", "sample_id"]:
                if col in df.columns:
                    vc = df[col].astype(str).value_counts(dropna=False)
                    row[f"top_{col}"] = vc.index[0] if len(vc) else "NA"
                    row[f"top_{col}_fraction"] = float(vc.iloc[0] / vc.sum()) if len(vc) else np.nan
            cnv_leiden_rows.append(row)
        cnv_leiden_summary = pd.DataFrame(cnv_leiden_rows)
    else:
        cnv_leiden_summary = pd.DataFrame()
    write_tsv_replace(cnv_leiden_summary, cnv_leiden_summary_tsv)

    # Per-cell scores, limited to essential columns.
    cell_cols = [
        CNV_GROUP_KEY, CLUSTER_KEY, "cnv_abs_mean_score", "core_dataset", "condition", "patient_id", "sample_id",
        "annotation_marker_based_HVG8000_r0_6", "lineage_marker_based_HVG8000_r0_6",
    ]
    if "cnv_leiden" in adata.obs.columns:
        cell_cols.append("cnv_leiden")
    if "cnv_cluster_score" in adata.obs.columns:
        cell_cols.append("cnv_cluster_score")
    cell_cols = [c for c in cell_cols if c in adata.obs.columns]
    cell_scores = adata.obs[cell_cols].copy()
    cell_scores.insert(0, "cell_id", adata.obs_names.astype(str))
    cell_scores.insert(0, "strategy", strategy)
    write_tsv_replace(cell_scores, cell_scores_tsv)

    # Figures.
    fig_paths = []
    try:
        if "cnv_umap" in adata.obsm:
            for color in [CNV_GROUP_KEY, CLUSTER_KEY, "cnv_abs_mean_score", "cnv_leiden"]:
                if color in adata.obs.columns:
                    fig = sc.pl.embedding(
                        adata,
                        basis="cnv_umap",
                        color=color,
                        show=False,
                        return_fig=True,
                        title=f"{strategy} | CNV UMAP | {color}",
                    )
                    fpath = CNV_FIGURE_DIR / f"{out_prefix}_cnv_umap_{color}.png"
                    fig.savefig(fpath, dpi=160, bbox_inches="tight")
                    plt.close(fig)
                    fig_paths.append(str(fpath))
    except Exception as e:
        print("[WARNING] CNV UMAP plotting failed:", repr(e))

    try:
        for groupby in [CNV_GROUP_KEY, CLUSTER_KEY]:
            if groupby in adata.obs.columns:
                cnv.pl.chromosome_heatmap_summary(
                    adata,
                    groupby=groupby,
                    use_rep="cnv",
                    figsize=(18, 8),
                    show=False,
                )
                fpath = CNV_FIGURE_DIR / f"{out_prefix}_chromosome_heatmap_summary_by_{groupby}.png"
                plt.savefig(fpath, dpi=160, bbox_inches="tight")
                plt.close("all")
                fig_paths.append(str(fpath))
    except Exception as e:
        print("[WARNING] chromosome heatmap summary failed:", repr(e))

    # Store run metadata in uns in a JSON-serializable way.
    adata.uns["infercnvpy_HVG8000_r0_6_run_info"] = {
        "strategy": strategy,
        "input_h5ad": str(input_h5ad),
        "preprocessing": preprocessing,
        "reference_key": CNV_GROUP_KEY,
        "reference_cat": REFERENCE_CAT,
        "window_size": WINDOW_SIZE,
        "step": STEP,
        "dynamic_threshold": DYNAMIC_THRESHOLD,
        "lfc_clip": LFC_CLIP,
        "chunksize": CHUNKSIZE,
        "n_jobs": N_JOBS,
        "exclude_chromosomes": list(EXCLUDE_CHROMOSOMES),
        "thresholds": thresholds,
    }

    write_h5ad_replace(adata, result_h5ad)

    elapsed_min = (time.time() - t0) / 60
    run_summary = pd.DataFrame([{
        "strategy": strategy,
        "input_h5ad": str(input_h5ad),
        "result_h5ad": str(result_h5ad),
        "cell_scores_tsv": str(cell_scores_tsv),
        "cluster_summary_tsv": str(cluster_summary_tsv),
        "cnv_leiden_summary_tsv": str(cnv_leiden_summary_tsv),
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_cnv_windows": int(adata.obsm["X_cnv"].shape[1]) if "X_cnv" in adata.obsm else np.nan,
        "preprocessing": preprocessing,
        "elapsed_minutes": round(elapsed_min, 3),
        "figures": ";".join(fig_paths),
        **thresholds,
    }])
    write_tsv_replace(run_summary, run_summary_tsv)

    # Free memory.
    del adata
    gc.collect()

    print(f"[DONE] {strategy} elapsed_min={elapsed_min:.2f}")
    return {
        "strategy": strategy,
        "run_summary_tsv": str(run_summary_tsv),
        "cluster_summary_tsv": str(cluster_summary_tsv),
        "cell_scores_tsv": str(cell_scores_tsv),
        "cnv_leiden_summary_tsv": str(cnv_leiden_summary_tsv),
        "result_h5ad": str(result_h5ad),
    }


In [ ]:

# =========================
# Run all selected strategies sequentially
# =========================

all_run_records = []

for _, row in manifest.iterrows():
    rec = run_one_infercnvpy_strategy(row)
    all_run_records.append(rec)

RUN_MANIFEST_TSV = METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv"
run_manifest = pd.DataFrame(all_run_records)
write_tsv_replace(run_manifest, RUN_MANIFEST_TSV)

display(run_manifest)


In [ ]:

# =========================
# Combine strategy-level outputs
# =========================

run_manifest = pd.read_csv(METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv", sep="\t")

cluster_summaries = []
cell_score_summaries = []
run_summaries = []
cnv_leiden_summaries = []

for _, r in run_manifest.iterrows():
    p = Path(r["cluster_summary_tsv"])
    if p.exists():
        cluster_summaries.append(pd.read_csv(p, sep="\t"))
    p = Path(r["run_summary_tsv"])
    if p.exists():
        run_summaries.append(pd.read_csv(p, sep="\t"))
    p = Path(r["cnv_leiden_summary_tsv"])
    if p.exists():
        cnv_leiden_summaries.append(pd.read_csv(p, sep="\t"))

combined_cluster = pd.concat(cluster_summaries, ignore_index=True) if cluster_summaries else pd.DataFrame()
combined_run = pd.concat(run_summaries, ignore_index=True) if run_summaries else pd.DataFrame()
combined_cnv_leiden = pd.concat(cnv_leiden_summaries, ignore_index=True) if cnv_leiden_summaries else pd.DataFrame()

COMBINED_CLUSTER_TSV = METADATA_DIR / "infercnvpy_combined_cluster_summary_HVG8000_r0_6.tsv"
COMBINED_RUN_TSV = METADATA_DIR / "infercnvpy_combined_run_summary_HVG8000_r0_6.tsv"
COMBINED_CNV_LEIDEN_TSV = METADATA_DIR / "infercnvpy_combined_cnv_leiden_summary_HVG8000_r0_6.tsv"

write_tsv_replace(combined_cluster, COMBINED_CLUSTER_TSV)
write_tsv_replace(combined_run, COMBINED_RUN_TSV)
write_tsv_replace(combined_cnv_leiden, COMBINED_CNV_LEIDEN_TSV)

display(combined_run)
display(combined_cluster.head())


In [ ]:

# =========================
# Consensus table across strategies
# =========================

if combined_cluster.empty:
    raise ValueError("combined_cluster is empty")

# Keep query/review clusters; references are used to estimate thresholds, not final malignant calls.
query_like = combined_cluster[combined_cluster["cnv_group_majority"].isin(["query", "review_query", "query_review", "query_putative_tumor_like"])].copy()

# In case the group values are exactly only reference/query in your inputs.
if query_like.empty:
    query_like = combined_cluster[combined_cluster["cnv_group_majority"].astype(str).str.contains("query", case=False, na=False)].copy()

if query_like.empty:
    print("[WARNING] No query-like clusters found by cnv_group_majority. Using all non-reference clusters.")
    query_like = combined_cluster[combined_cluster["cnv_group_majority"] != "reference"].copy()

call_strength = {
    "CNV_low_or_reference_like": 0,
    "CNV_intermediate_candidate": 1,
    "CNV_high_candidate": 2,
}
query_like["cnv_call_strength"] = query_like["cnv_call"].map(call_strength).fillna(0).astype(int)

cons_rows = []
for cl, df in query_like.groupby("cluster", observed=True):
    n_strat = df["strategy"].nunique()
    n_high = int((df["cnv_call"] == "CNV_high_candidate").sum())
    n_intermediate_or_high = int((df["cnv_call_strength"] >= 1).sum())
    strategies_high = ",".join(df.loc[df["cnv_call"] == "CNV_high_candidate", "strategy"].astype(str).tolist())
    strategies_intermediate_or_high = ",".join(df.loc[df["cnv_call_strength"] >= 1, "strategy"].astype(str).tolist())
    median_score_median = float(df["median_cnv_abs_score"].median())
    max_effect = float(df["robust_effect_vs_reference_median"].max())

    # Consensus logic.
    main_core = df[df["strategy"].isin(["core_conservative", "core_plus_lymphoid", "extended_myeloid"])]
    core_n = main_core.shape[0]
    core_high = int((main_core["cnv_call"] == "CNV_high_candidate").sum())
    core_int_high = int((main_core["cnv_call_strength"] >= 1).sum())

    if core_n > 0 and core_high >= 2:
        consensus = "robust_CNV_high_candidate"
    elif core_n > 0 and core_int_high >= 2:
        consensus = "recurrent_CNV_intermediate_candidate"
    elif n_high >= 1:
        consensus = "single_strategy_CNV_high_review"
    elif n_intermediate_or_high >= 1:
        consensus = "single_strategy_CNV_intermediate_review"
    else:
        consensus = "CNV_low_or_reference_like"

    # Carry forward annotation columns if present.
    row = {
        "cluster": str(cl),
        "n_strategies_observed": int(n_strat),
        "n_high_calls": n_high,
        "n_intermediate_or_high_calls": n_intermediate_or_high,
        "strategies_high": strategies_high,
        "strategies_intermediate_or_high": strategies_intermediate_or_high,
        "median_of_strategy_median_cnv_abs_score": median_score_median,
        "max_robust_effect_vs_reference_median": max_effect,
        "consensus_cnv_call": consensus,
    }
    for col in [
        "top_annotation_marker_based_HVG8000_r0_6",
        "top_lineage_marker_based_HVG8000_r0_6",
        "top_cnv_role_HVG8000_r0_6",
        "top_bias_status_HVG8000_r0_6",
    ]:
        if col in df.columns:
            vc = df[col].astype(str).value_counts(dropna=False)
            row[col] = vc.index[0] if len(vc) else "NA"
    cons_rows.append(row)

consensus = pd.DataFrame(cons_rows).sort_values([
    "consensus_cnv_call", "max_robust_effect_vs_reference_median"
], ascending=[True, False])

CONSENSUS_TSV = METADATA_DIR / "infercnvpy_consensus_cluster_calls_HVG8000_r0_6.tsv"
write_tsv_replace(consensus, CONSENSUS_TSV)

display(consensus)


In [ ]:

# =========================
# Output manifest
# =========================

items = {
    "readiness_report": METADATA_DIR / "infercnvpy_run_readiness_HVG8000_r0_6.tsv",
    "strategy_run_manifest": METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv",
    "combined_run_summary": METADATA_DIR / "infercnvpy_combined_run_summary_HVG8000_r0_6.tsv",
    "combined_cluster_summary": METADATA_DIR / "infercnvpy_combined_cluster_summary_HVG8000_r0_6.tsv",
    "combined_cnv_leiden_summary": METADATA_DIR / "infercnvpy_combined_cnv_leiden_summary_HVG8000_r0_6.tsv",
    "consensus_cluster_calls": METADATA_DIR / "infercnvpy_consensus_cluster_calls_HVG8000_r0_6.tsv",
}

# Add per-strategy outputs from run_manifest.
if (METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv").exists():
    rm = pd.read_csv(METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv", sep="\t")
    for _, r in rm.iterrows():
        strategy = r["strategy"]
        for col in ["run_summary_tsv", "cluster_summary_tsv", "cell_scores_tsv", "cnv_leiden_summary_tsv", "result_h5ad"]:
            if col in rm.columns:
                items[f"{strategy}_{col}"] = Path(r[col])

rows = []
for item, path in items.items():
    path = Path(path)
    rows.append({
        "item": item,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan,
    })

out_manifest = pd.DataFrame(rows)
OUT_MANIFEST_TSV = METADATA_DIR / "infercnvpy_output_manifest_HVG8000_r0_6.tsv"
write_tsv_replace(out_manifest, OUT_MANIFEST_TSV)

display(out_manifest)



## Files to send back after execution

Send these TSV files back for review:

```text
${GLIOMA_PROJECT_ROOT}/metadata/infercnvpy_run_readiness_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/infercnvpy_combined_run_summary_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/infercnvpy_combined_cluster_summary_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/infercnvpy_combined_cnv_leiden_summary_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/infercnvpy_consensus_cluster_calls_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/infercnvpy_output_manifest_HVG8000_r0_6.tsv
```

If upload size is not a problem, also send the five per-strategy cluster summaries and run summaries listed in the output manifest. Do **not** send `.h5ad` files.
